Création de la liste d'adjacence :

Import des données
Liste des noms des station
Liste des coordonnées
Triangulation de Delauney
Création de la liste
Ajout des connexion dans la liste 
Export en JSON

In [2]:
import json
import numpy as np
from scipy.spatial import Delaunay
import os

# Définir le chemin du fichier dans le dossier parent
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))

# Charger les données depuis le fichier JSON
with open(file_path, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Extraire les noms des stations et leurs coordonnées (lat, lon)
nom_station = [f"{s['name']} ({s['stationCode']})" for s in stations]  # Liste des noms
coo = np.array([[s["lat"], s["lon"]] for s in stations])  # Coordonnées

# Effectuer la Tri de Delaunay
Triangulation = Delaunay(coo)

# Initialiser la liste d'adjacence
Liste_adjacence = {name: set() for name in nom_station}  # Dictionnaire de sets

# Remplir la liste d'adjacence à partir des Triangles
for simplex in Triangulation.simplices:  # simplex = un Triangle (3 sommets)
    for i in range(3):  # Parcourir les 3 sommets du Triangle
        for j in range(3):
            if i != j:  # Ne pas ajouter une station comme voisine d'elle-même
                station_i = nom_station[simplex[i]]
                station_j = nom_station[simplex[j]]
                Liste_adjacence[station_i].add(station_j)

# Convertir les sets en listes pour un affichage JSON-friendly
Liste_adjacence = {k: list(v) for k, v in Liste_adjacence.items()}

# Afficher un extrait de la liste d'adjacence
print(json.dumps(Liste_adjacence, indent=4, ensure_ascii=False))

# Sauvegarder la liste d'adjacence dans un fichier JSON
with open("velib_Liste_adjacence.json", "w", encoding="utf-8") as f:
    json.dump(Liste_adjacence, f, indent=4, ensure_ascii=False)


{
    "Benjamin Godard - Victor Hugo (16107)": [
        "Mairie du 16ème (16013)",
        "Flandrin - Longchamp (16012)",
        "Victor Hugo - La Pompe (16011)",
        "Flandrin - Henri Martin (16018)"
    ],
    "Hôpital Mondor (40001)": [
        "Iles de Loisirs de Créteil (40015)",
        "Liberté - Vert-de-Maisons (47007)",
        "Centre Hospitalier Intercommunal de Créteil (40003)",
        "Créteil Village (40004)",
        "Préfecture de Créteil (40007)",
        "Les Juilliottes (47002)",
        "Bleuets - Bordières (40002)"
    ],
    "Rouget de L'isle - Watteau (44015)": [
        "Lebrun - Colonel Fabien  (44011)",
        "Youri Gagarine - Commune de Paris (44008)",
        "Conservatoire de Musique (46003)",
        "Camille Risch - Paul Armangot (44014)",
        "8 Mai 1945 - 10 Juillet 1940 (44010)",
        "Balzac - Olympes de Gouges (44016)"
    ],
    "Toudouze - Clauzel (9020)": [
        "Jean-Baptiste Pigalle - La Bruyere (9026)",
        "Choron - Mar

SAE

Import des librairies et fichiers

In [3]:
import folium
import json
import numpy as np
from scipy.spatial import Delaunay, Voronoi
from folium.plugins import MarkerCluster
import branca.colormap as cm
from matplotlib import colors as mcolors
import matplotlib.pyplot as plt
from math import radians, sin, cos, sqrt, atan2
import os

# Définir le chemin du fichier velib
file_path_velib = os.path.abspath(os.path.join(os.getcwd(), "..", "velib_stations.json"))
# Définir le chemin de la liste d'adjacence
file_path_adj = "velib_Liste_adjacence.json"

# Charger les données depuis le fichier JSON
with open(file_path_velib, "r", encoding="utf-8") as file:
    stations = json.load(file)["data"]["stations"]

# Charger la liste d'adjacence
with open(file_path_adj, "r") as f:
    adj_list = json.load(f)

Fonction Indice repartition

In [4]:
def repartition(Nv, Cmax, C):
    if Nv == 6:
        return 0
    else:
        return 0.5 * ((Nv - 6)/6) + (1 - 0.5) * ((Cmax - C)/Cmax)

Fonction Calcule distance eulerienne cable

In [5]:
def longueur_cable(lat1, lon1, lat2, lon2):
    R = 6371  # Rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c  # Distance en km

Liste des points des stations

In [6]:
# Extraire les coordonnées (latitude, longitude)
points = np.array([[s["lat"], s["lon"]] for s in stations])

Triangulation de Delauney + Cellules de Voronoi

In [7]:
# Effectuer la Triangulation de Delaunay et Voronoi
Triangulation = Delaunay(points)
Vor = Voronoi(points, qhull_options='Qbb Qc Qx')

Eviter les doublons et une liste avec le nom et les coordonnée

In [8]:
# On crée une colonne id_nom pour eviter les doublons de station
for s in stations:
    s["id_nom"] = f"{s['name']} ({s['stationCode']})"

# Création d'un dictionnaire de correspondance nom -> coordonnées
station_coords = {s["id_nom"]: (s["lat"], s["lon"]) for s in stations}

Construire la Liste des arretes et tri

In [9]:
# Construire la liste des arêtes pondérées
aretes = []
for station, voisins in adj_list.items():
    if station in station_coords:
        lat1, lon1 = station_coords[station]
        for voisin in voisins:
            if voisin in station_coords:
                lat2, lon2 = station_coords[voisin]
                distance = longueur_cable(lat1, lon1, lat2, lon2)
                aretes.append((distance, station, voisin))

# Trier les arêtes par poids croissant
aretes.sort()

Algorithme de kruskal

In [10]:
# Algorithme de Kruskal
class Kruskal:
    def __init__(self, elements):
        self.parent = {e: e for e in elements}
        self.rang = {e: 0 for e in elements}

    def Trouver(self, objet):
        if self.parent[objet] != objet:
            self.parent[objet] = self.Trouver(self.parent[objet])  # Compression de chemin
        return self.parent[objet]

    def RemonterArbre(self, station, voisin):
        root1 = self.Trouver(station)
        root2 = self.Trouver(voisin)
        if root1 != root2:
            if self.rang[root1] > self.rang[root2]:
                self.parent[root2] = root1
            elif self.rang[root1] < self.rang[root2]:
                self.parent[root1] = root2
            else:
                self.parent[root2] = root1
                self.rang[root1] += 1

# Initialiser Kruskal
stations_names = list(station_coords.keys())
krusk = Kruskal(stations_names)
ArbreCouvrantMinimal = []

for weight, station, voisin in aretes:
    if krusk.Trouver(station) != krusk.Trouver(voisin):
        krusk.RemonterArbre(station, voisin)
        ArbreCouvrantMinimal.append((station, voisin, weight))
        if len(ArbreCouvrantMinimal) == len(stations_names) - 1:
            break

Carte Folium + ajout des points + Triangulation + ACM + Cellules de Voronoi

In [11]:
# Centrer la carte sur Paris
paris_coords = (48.8566, 2.3522)
m = folium.Map(location=paris_coords, zoom_start=12, tiles="CartoDB positron")
cluster = MarkerCluster(name="Station Cluster").add_to(m)

# Définition d'un colormap (rouge → jaune → vert)
colormap = cm.LinearColormap(
    colors=["green", "yellow", "red"],  # Dégradé
    vmin=-0, vmax=1,  # Intervalle de l'indice Ir
    caption="Indice de Performance"
)

# Définition d'un colormap (noir → gris → blanc)
colormapVor = cm.LinearColormap(
    colors=["#FFFFFF", "#8b8b8b", "#000000"],  # Dégradé
    vmin=-0, vmax=1,  # Intervalle de l'indice Ir
    caption="Voronoi"
)

Cmax = max(station["capacity"] for station in stations)


# Ajouter le nombre de voisins pour chaque station
for s in stations:
    station_name = s["id_nom"]
    if station_name in adj_list:
        s["Nv"] = len(adj_list[station_name])
    else:
        s["Nv"] = 0
    s["Ir"] = repartition(s["Nv"], Cmax, s["capacity"])

# Coloration des cellules Voronoi (y compris infinies)
for idx, region_index in enumerate(Vor.point_region):
    region = Vor.regions[region_index]
    if not -1 in region and len(region) > 0:
        polygon = [Vor.vertices[i] for i in region]
        colorvor = colormapVor(stations[idx]["Ir"])
        folium.Polygon(
            locations=[(lat, lon) for lat, lon in polygon],
            color=None,
            fill=True,
            fill_opacity=0.6,
            fill_color=colorvor,
            tooltip=f"{stations[idx]['name']}<br>Ir = {stations[idx]['Ir']:.2f}"
        ).add_to(m)

# Ajouter les stations Vélib' sur la carte
for s in stations:
    color = colormap(s["Ir"])  # Associe la couleur en fonction de I_r
    folium.CircleMarker(
        location=[s["lat"], s["lon"]],
        popup=f"Station: {s['name']}<br>Capacité: {s['capacity']}<br>Indice Ir: {s['Ir']:.2f}" ,max_width=250,
        color=color,
        radius=s["capacity"]//4,
        fill=True,
        fill_color=color
    ).add_to(cluster)
    
# Ajouter les Triangles de Delaunay sous forme de lignes
for simplex in Triangulation.simplices:
    point = [points[i] for i in simplex]  # Récupérer les sommets du Triangle
    folium.PolyLine(locations=point + [point[0]], color="red", weight=2).add_to(m)  # Fermer le Triangle

# Ajouter les arêtes de l'ACM
for u, v, weight in ArbreCouvrantMinimal:
    lat1, lon1 = station_coords[u]
    lat2, lon2 = station_coords[v]
    folium.PolyLine([(lat1, lon1), (lat2, lon2)], color="green", weight=4).add_to(m)

Création page HTML

In [12]:

# Ajouter une légende personnalisée
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    right: 50px;
    background-color: white;
    padding: 10px;
    border-radius: 8px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    font-size: 14px;
    z-index: 999;
">
    <b>Légende :</b><br>
    🚲 <span style="color:blue;">Points Bleus</span> - Stations Vélib'<br>
    🔺 <span style="color:red;">Lignes Rouges</span> - Triangulation de Delaunay<br>
    🟢 <span style="color:green;">Arbre Couvrant Minimal</span> - ACM
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("velib_delaunay_legend.html")